In [13]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

## Load database

In [14]:
# Setup pandas display options
pd.set_option('display.max_columns', None)

BASE_PATH = Path.cwd().parent
DATA_PATH = BASE_PATH / "researchment" / "database" / "indonesia_fintech_paylater.parquet"

# Load the dataset
df = pd.read_parquet(DATA_PATH)
df.head()

,borrower_id,age,gender,province,education_level,employment_status,monthly_income_idr,regional_minimum_wage_umr_idr,num_active_lending_apps,total_loan_outstanding_idr,credit_limit_granted_idr,loan_tenor_months,daily_interest_rate_pct,primary_loan_purpose,debt_to_income_ratio,income_to_umr_ratio,credit_limit_utilization_ratio,monthly_installment_estimate_idr,debt_service_ratio,disposable_income_after_debt_idr,days_past_due,financial_distress_index,twp90_default_status,primary_default_root_cause,cot_financial_psychology_rationale,confidence_score,risk_score,risk_short,is_high_risk,distress_bin
0,ID_DEB_00264B68D1,24,Female,Bali,High School / SMK,Micro-Entrepreneur / UMKM,6420000,2950000,2,8380000,12280000,1,0.35,Daily Living & Emergency Buffer,1.31,2.18,0.6824,9259900.0,1.4424,-2839900.0,67,3.97,Doubtful-DPD61-90,High-Interest-Rate-Compounding,"Debitur 'ID_DEB_00264B68D1' (Female, usia 24 t...",0.961,3,Doubt(61-90),True,"(3.686, 4.588]"
1,ID_DEB_0DE3583B12,28,Female,Bali,Bachelor (S1),Formal Private Employee,8630000,2950000,1,20110000,29060000,6,0.30,Daily Living & Emergency Buffer,2.33,2.93,0.6920,5161567.0,0.5981,3468433.0,11,4.87,Special-Mention-DPD1-30,High-Interest-Rate-Compounding,"Debitur 'ID_DEB_0DE3583B12' (Female, usia 28 t...",0.975,1,SM(1-30),False,"(4.588, 5.49]"
2,ID_DEB_0F661FADC0,20,Female,Bali,High School / SMK,Civil Servant / BUMN,5720000,2950000,4,14960000,23900000,1,0.32,Daily Living & Emergency Buffer,2.62,1.94,0.6259,16396160.0,2.8665,-10676160.0,180,6.07,Default-Macet-TWP90,High-Interest-Rate-Compounding,"Debitur 'ID_DEB_0F661FADC0' (Female, usia 20 t...",0.990,4,Default Rate(>90),True,"(5.49, 6.392]"
3,ID_DEB_10C74D3F21,31,Female,Bali,Bachelor (S1),Civil Servant / BUMN,11780000,2950000,3,7620000,26130000,1,0.26,Daily Living & Emergency Buffer,0.65,3.99,0.2916,8214360.0,0.6973,3565640.0,24,2.35,Special-Mention-DPD1-30,Compulsive-FOMO-Spending,"Debitur 'ID_DEB_10C74D3F21' (Female, usia 31 t...",0.977,1,SM(1-30),False,"(1.882, 2.784]"
4,ID_DEB_12CFCAE01F,22,Male,Bali,Bachelor (S1),Formal Private Employee,6490000,2950000,2,11080000,16420000,9,0.23,Daily Living & Emergency Buffer,1.71,2.20,0.6748,1995631.0,0.3075,4494369.0,0,4.43,Current-Performing,Compulsive-FOMO-Spending,"Debitur 'ID_DEB_12CFCAE01F' (Male, usia 22 tah...",0.980,0,Current,False,"(3.686, 4.588]"


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 30 columns):
 #   Column                              Non-Null Count  Dtype   
---  ------                              --------------  -----   
 0   borrower_id                         10000 non-null  object  
 1   age                                 10000 non-null  int64   
 2   gender                              10000 non-null  object  
 3   province                            10000 non-null  object  
 4   education_level                     10000 non-null  object  
 5   employment_status                   10000 non-null  object  
 6   monthly_income_idr                  10000 non-null  int64   
 7   regional_minimum_wage_umr_idr       10000 non-null  int64   
 8   num_active_lending_apps             10000 non-null  int64   
 9   total_loan_outstanding_idr          10000 non-null  int64   
 10  credit_limit_granted_idr            10000 non-null  int64   
 11  loan_tenor_months            

### - **🎯 Task 1: OJK TWP90 Credit Default Early-Warning Modeling**
   #### - **Objective:** Target a borrower which cross the the 90-day non-performing loan in **`twp90_default_status`** column to analysis what is factor behind it.
        Train (LightGBM, XGBoost, CatBoost) Models  to predict whether a borrower will cross the 90-day non-performing loan threshold (`Default-Macet-TWP90` vs Performing).
   ##### - **Recommended ML Models:** LightGBM, Random Forest, XGBoost, CatBoost
   ##### - **Evaluation Metrics:** `Precision-Recall AUC (PR-AUC), Brier Calibration Score, Macro F1-Score, ROC-AUC`


## Feature engineering

In [16]:
# Add datetime features on df
np.random.seed(42)

# Define a realistic time window (e.g. past 2 years up to today)
start_date = pd.to_datetime('2024-11-05')
end_date = pd.to_datetime('2026-11-05')

# Generate 10000 random timestamps between these dates
random_timestamps = np.random.randint(int(start_date.timestamp()), int(end_date.timestamp()), size=len(df))

# Add timestamp features to the DataFrame
df['date'] = pd.to_datetime(random_timestamps, unit='s')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['hour'] = df['date'].dt.strftime('%H:%M:%S')

# Move the date column to the first position
date_col = "date"
columns_move = df.columns.tolist()
columns_move.remove(date_col)

borrower_index = columns_move.index("borrower_id")
columns_move.insert(borrower_index + 1, date_col)

df = df[columns_move]

# Sort the DataFrame by the new 'date' column
df = df.sort_values(by='date').reset_index(drop=True)

In [17]:
# Build X features
FEATURES = df.drop(columns=['borrower_id', 'date', 'year', 'month', 'day', 'hour'])

## Encode data

In [18]:
df_encoded = df.copy()

# Encoding the categorical, object, boolean
from sklearn.preprocessing import LabelEncoder

for col in df_encoded.select_dtypes(include=['object', 'category', 'bool']).columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

## Split data into X and y

In [19]:
from sklearn.model_selection import train_test_split

# Define features on encoded DataFrame
FEATURES_ENCODED = df_encoded.drop(columns=['borrower_id', 'date', 'year', 'month', 'day', 'hour', 'twp90_default_status', 'risk_short', 'risk_score', 'days_past_due'])

X = FEATURES_ENCODED
y = df_encoded['twp90_default_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Scale data

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)

X_train shape: (8000, 25)
X_test shape: (2000, 25)


## Evaluating before training ML models

In [21]:
from sklearn.feature_selection import mutual_info_classif

target_col = 'twp90_default_status'

mi = mutual_info_classif(X_train_scaled, y_train, random_state=42)
mi_df = pd.DataFrame({"feature": X_train.columns, "mutual_info": mi}).sort_values(by="mutual_info", ascending=False)

print("Top 10 features by mutual information:")
print(mi_df.head(10).to_string(index=False))

# 🚩 Anything > 0.9 is almost certainly a leak
print("\n🚩 SUSPECTED LEAKS (MI > 0.9):")
print(mi_df[mi_df["mutual_info"] > 0.9])

Top 10 features by mutual information:
                         feature  mutual_info
                    is_high_risk     0.642768
                confidence_score     0.628957
        financial_distress_index     0.551700
                    distress_bin     0.537491
              debt_service_ratio     0.437660
  credit_limit_utilization_ratio     0.363769
            debt_to_income_ratio     0.354682
disposable_income_after_debt_idr     0.337559
            primary_loan_purpose     0.287533
monthly_installment_estimate_idr     0.261450

🚩 SUSPECTED LEAKS (MI > 0.9):
Empty DataFrame
Columns: [feature, mutual_info]
Index: []


## Train used to ML Models -> LightGBM, XGBoost, CatBoost, Random Forest ML Models chosen

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, precision_score

# Define ml models bundle
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0)
}

# Define hyperparameter grids for each model
param_search = {
    "Random Forest": {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    "XGBoost": {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    },
    "LightGBM": {
        'n_estimators': [100, 200, 300],
        'max_depth': [-1, 10, 20],
        'learning_rate': [0.01, 0.1, 0.2],
        'num_leaves': [31, 50, 100],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    },
    "CatBoost": {
        'iterations': [100, 200, 300],
        'depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'l2_leaf_reg': [1, 3, 5],
        'border_count': [32, 64, 128]
    }
}

# -- Function to perform hyperparameter tuning and model evaluation -- 

results = []
best_models = {}

def tune_and_evaluate_model(
    model_name,
    model,
    param_grid,
    X_train,
    y_train,
    X_test,
    y_test
):
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=10,
        scoring="roc_auc_ovr_weighted",
        cv=5,
        random_state=42,
        n_jobs=-1,
        verbose=1,
        error_score='raise'
    )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    best_params = search.best_params_

    # Predictions
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)

    # Evaluation metrics
    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, average='weighted', zero_division=0),
        "ROC AUC": roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    }

    best_models[model_name] = best_model
    best_params[model_name] = best_params
    results.append(result)

    print(f"\n{model_name}")
    print(f"Accuracy: {result['Accuracy']:.4f}")
    print(f"Weighted F1: {result['F1 Score']:.4f}")
    print(f"ROC AUC: {result['ROC AUC']:.4f}")

    return best_model, result

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

best_models = {}
all_results = []

for model_name, model in tqdm(
    models.items(),
    total=len(models),
    desc="Training models"
):
    trained_model, result = tune_and_evaluate_model(
        model_name=model_name,
        model=model,
        param_grid=param_search[model_name],
        X_train=X_train_scaled,
        y_train=y_train,
        X_test=X_test_scaled,
        y_test=y_test
    )

    best_models[model_name] = trained_model
    all_results.append(result)

# Result table
result_df = (
    pd.DataFrame(all_results)
    .sort_values(by="ROC AUC", ascending=False)
    .reset_index(drop=True)
)

result_df

Training models:   0%|          | 0/4 [00:00<?, ?it/s]

Fitting 5 folds for each of 10 candidates, totalling 50 fits
